# LangGraph Customer Support Agent with Snowflake AI Observability

This notebook demonstrates how to:

1. **Build a LangGraph agent** that uses Snowflake Cortex Search, Cortex Analyst, and chart generation as tools
2. **Instrument the agent with TruLens** (`TruGraph`) for full trace capture
3. **Run evaluations** using Snowflake AI Observability to measure answer relevance, context relevance, and groundedness

### Architecture

```
User Query
    │
    ▼
LangGraph ReAct Agent (ChatSnowflake LLM)
    ├── Tool: Cortex Search (unstructured case lookups)
    ├── Tool: Cortex Analyst (structured metrics via semantic view)
    └── Tool: Generate Chart (Vega-Lite visualizations via Cortex LLM)
    │
    ▼
TruGraph (auto-instrumentation) → Snowflake AI Observability
```

### Prerequisites

- Run `setup.sql` to create the `CUST_SUPPORT_DEMO.SUPPORT` schema, tables, semantic view, and Cortex Search service
- Snowflake account with Cortex features enabled
- `CORTEX_USER` database role and `CREATE EXTERNAL AGENT` / `CREATE TASK` / `EXECUTE TASK` privileges

In [1]:
%pip install --quiet -U langchain-core langchain-snowflake langgraph trulens-core trulens-providers-cortex trulens-connectors-snowflake trulens-apps-langgraph altair

Note: you may need to restart the kernel to use updated packages.


## 1. Snowflake Session Setup

In [2]:
import os

sorted(os.environ)[-30:]

['PYTHONUNBUFFERED',
 'PYTHON_FROZEN_MODULES',
 'SHELL',
 'SHLVL',
 'SNOWFLAKE_ACCOUNT',
 'SNOWFLAKE_PASSWORD',
 'SNOWFLAKE_PAT',
 'SNOWFLAKE_USER',
 'SNOWFLAKE_USER_PASSWORD',
 'SSH_AUTH_SOCK',
 'TERM',
 'TMPDIR',
 'USER',
 'VSCODE_CODE_CACHE_PATH',
 'VSCODE_CRASH_REPORTER_PROCESS_TYPE',
 'VSCODE_CWD',
 'VSCODE_ESM_ENTRYPOINT',
 'VSCODE_HANDLES_UNCAUGHT_ERRORS',
 'VSCODE_IPC_HOOK',
 'VSCODE_L10N_BUNDLE_LOCATION',
 'VSCODE_NLS_CONFIG',
 'VSCODE_PID',
 'XPC_FLAGS',
 'XPC_SERVICE_NAME',
 '_',
 '_CE_CONDA',
 '_CE_M',
 '__CFBundleIdentifier',
 '__CF_USER_TEXT_ENCODING',
 '__CONDA_SHLVL_1_SNOWFLAKE_PASSWORD']

In [3]:
import os

#Define env variables as necessary 

# os.environ["SNOWFLAKE_ACCOUNT"] = "<your_account>"
# os.environ["SNOWFLAKE_USER"] = "<your_username>"
# os.environ["SNOWFLAKE_PASSWORD"] = "<your_password>"
os.environ["SNOWFLAKE_WAREHOUSE"] = "COMPUTE_WH"
os.environ["SNOWFLAKE_DATABASE"] = "CUST_SUPPORT_DEMO"
os.environ["SNOWFLAKE_SCHEMA"] = "AGENTS"

from langchain_snowflake import create_session_from_env

session = create_session_from_env()
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

/opt/anaconda3/envs/snow_tru_py314/lib/python3.11/site-packages/langchain_snowflake/chat_models/base.py:26: UserWarning: Field name "schema" in "ChatSnowflake" shadows an attribute in parent "BaseChatModel"
  class ChatSnowflake(


Connected: "CUST_SUPPORT_DEMO"."AGENTS"


## 2. Define Snowflake Cortex Tools

In [4]:
import json
import textwrap
from langchain_core.tools import tool
from langchain_snowflake import SnowflakeCortexSearchRetriever

from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

CORTEX_SEARCH_SERVICE = "CUST_SUPPORT_DEMO.AGENTS.CASE_SEARCH_SERVICE"

retriever = SnowflakeCortexSearchRetriever(
    session=session,
    schema="CUST_SUPPORT_DEMO.AGENTS",
    service_name=CORTEX_SEARCH_SERVICE,
    k=10,
    auto_format_for_rag=True,
    content_field="ISSUE_SUMMARY",
    join_separator="\n\n",
    fallback_to_page_content=True,
)


@tool
@instrument(
    span_type=SpanAttributes.SpanType.RETRIEVAL,
    attributes={
        SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
        SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
    },
)
def search_support_cases(query: str) -> list:
    """ Search specific case details, issue descriptions, and resolution summaries by topic or keyword.
        Returns CASE_ID, CUSTOMER_ID, PRODUCT, ISSUE_CATEGORY, PRIORITY, STATUS, REP_NAME as attributes.
        Always cite CASE_ID values in responses. NOT for aggregate metrics or trends"""
    docs = retriever.invoke(query)
    # return "\n\n".join(doc.page_content for doc in docs)
    return [doc.page_content for doc in docs]

print("Cortex Search tool defined.")
test_results = search_support_cases.invoke("authentication login failures")
print(test_results)

Cortex Search tool defined.
['Customer unable to log in after password reset. MFA token not being accepted despite correct entry. Resolution: Reset MFA configuration and re-enrolled the user. Cleared cached authentication tokens on the server side. Verified successful login after re-enrollment.', 'Customer unable to log in after password reset. MFA token not being accepted despite correct entry. Resolution: Reset MFA configuration and re-enrolled the user. Cleared cached authentication tokens on the server side. Verified successful login after re-enrollment.', 'Customer unable to log in after password reset. MFA token not being accepted despite correct entry. Resolution: Reset MFA configuration and re-enrolled the user. Cleared cached authentication tokens on the server side. Verified successful login after re-enrollment.', 'Customer unable to log in after password reset. MFA token not being accepted despite correct entry. Resolution: Reset MFA configuration and re-enrolled the user. C

In [9]:
import requests

SEMANTIC_VIEW = "CUST_SUPPORT_DEMO.AGENTS.SUPPORT_ANALYTICS"


@tool
@instrument(
    span_type=SpanAttributes.SpanType.RETRIEVAL,
    attributes={
        SpanAttributes.RETRIEVAL.QUERY_TEXT: "question",
        SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
    },
)
def query_support_analytics(question: str) -> list:
    """ Quantitative support metrics: case counts, CSAT scores, resolution times, first response
        times, escalation rates, rep performance, daily/weekly trends, breakdowns by product/category/priority/rep.
        Use for aggregation, filtering, or numerical analysis. NOT for case descriptions or resolutions."""
    host = session.connection.host
    token = session.connection.rest.token
    # token = os.getenv("SNOWFLAKE_PAT")

    resp = requests.post(
        url=f"https://{host}/api/v2/cortex/analyst/message",
        json={
            "messages": [
                {"role": "user", "content": [{"type": "text", "text": question}]}
            ],
            "semantic_view": "CUST_SUPPORT_DEMO.AGENTS.SUPPORT_ANALYTICS",
        },
        headers={
            "Authorization": f'Snowflake Token="{token}"',
            "Content-Type": "application/json",
        },
    )
    resp.raise_for_status()
    data = resp.json()

    result_parts = []
    msg = data.get("message") or {}
    for block in msg.get("content", []):
        if block.get("type") == "text":
            result_parts.append(block["text"])
        elif block.get("type") == "sql":
            result_parts.append(f"SQL: {block['statement']}")
            try:
                sql_results = session.sql(block["statement"]).collect()
                result_parts.append(json.dumps([row.as_dict() for row in sql_results[:20]], default=str))
            except Exception as e:
                result_parts.append(f"SQL execution error: {e}")
        elif block.get("type") == "result_table":
            result_parts.append(json.dumps(block.get("data", [])[:20], default=str))
    # return "\n".join(result_parts) if result_parts else "No results returned."
    return result_parts

print("Cortex Analyst tool defined.")
test_analytics = query_support_analytics.invoke("How many total support cases are there?")
print(test_analytics[:300])

Cortex Analyst tool defined.
['This is our interpretation of your question:\n\nHow many total support cases are there over the entire available time period?', 'SQL: WITH __case_metrics AS (\n  SELECT\n    case_id,\n    case_date\n  FROM CUST_SUPPORT_DEMO.AGENTS.CASE_METRICS\n)\nSELECT\n  MIN(case_date) AS start_date,\n  MAX(case_date) AS end_date,\n  COUNT(case_id) AS total_cases\nFROM __case_metrics\n -- Generated by Cortex Analyst (request_id: 53dd852a-6591-4b84-8199-afeac6fdcac4)\n;', '[{"START_DATE": "2025-12-01", "END_DATE": "2026-02-28", "TOTAL_CASES": 200}]']


In [10]:
import altair as alt
from snowflake.cortex import complete


@tool
def generate_chart(data: list[dict], chart_description: str) -> str:
    """Generate a Vega-Lite chart visualization from data.
    Use after query_support_analytics returns tabular results that would benefit
    from visual representation (trends, comparisons, distributions, rankings).

    Args:
        data: List of dictionaries representing rows of data (from SQL query results).
        chart_description: Natural language description of the desired chart.
    """
    prompt = f"""Generate a Vega-Lite v5 JSON specification for the following:
Description: {chart_description}
Data: {json.dumps(data[:50], default=str)}

Rules:
- Return ONLY valid JSON, no markdown fences or explanation.
- Set $schema to \"https://vega.github.io/schema/vega-lite/v5.json\".
- Embed data inline using \"data\": {{\"values\": [...]}}.
- Use a SINGLE mark type. Do NOT use \"layer\". Keep the spec simple.
- Include title, mark, encoding. Use width of 500 and height of 300."""

    spec_text = complete(
        model="claude-4-sonnet",
        prompt=prompt,
        session=session,
    )

    if "```" in spec_text:
        spec_text = spec_text.split("```")[1]
        if spec_text.startswith("json"):
            spec_text = spec_text[4:]
        spec_text = spec_text.strip()

    chart_spec = json.loads(spec_text)

    # Remove 'layer' if present alongside 'mark' to avoid altair v6 validation errors
    if "layer" in chart_spec and "mark" in chart_spec:
        del chart_spec["layer"]

    chart = alt.Chart.from_dict(chart_spec, validate=False)
    display(chart)

    return json.dumps(chart_spec)


print("Chart generation tool defined.")

Chart generation tool defined.


## 3. Build the LangGraph Agent

In [11]:
from langchain_snowflake import ChatSnowflake
from langgraph.prebuilt import create_react_agent

llm = ChatSnowflake(
    session=session,
    model="claude-sonnet-4-6",
    temperature=0,
    max_tokens=2048,
)

SYSTEM_PROMPT = """
instructions:
  response: >
    Concise customer support analytics assistant. Be data-driven, use appropriate number formatting.
    ALWAYS cite case IDs (e.g., CASE-00016) from search results. For rankings/comparisons, show the
    COMPLETE list. Group duplicate search results by issue type with counts and representative case IDs.
    Note data limitations when results appear incomplete.
  orchestration: >
    Tool Selection: query_support_analytics for metrics, trends, counts, averages, comparisons.
    search_support_cases for specific case details, issue descriptions, resolutions, or topic searches.
    generate_chart for visualizing tabular data from query_support_analytics (trends, comparisons,
    distributions, rankings). Use BOTH search and analytics tools when a question needs metrics AND
    case examples.

    search_support_cases: ALWAYS include CASE_ID from results. Group duplicate results by issue type
    with representative case IDs rather than repeating identical entries.

    query_support_analytics: For trends, request the FULL date range (data covers Dec 2025-Feb 2026).
    For rankings, return ALL items with values, not just the top result.

    generate_chart: After getting tabular data from query_support_analytics, call generate_chart with
    the raw data rows and a description of the desired visualization. Use for any result with 3+ rows
    that shows trends, comparisons, or distributions.
  sample_questions:
    - question: "What is the average CSAT score by product?"
      answer: "I'll query the support analytics data to get average CSAT scores for all products, present the complete ranking, and generate a bar chart."
    - question: "Find cases related to authentication issues"
      answer: "I'll search the case database for authentication-related cases and cite specific case IDs grouped by issue type."
    - question: "Which rep has the best resolution time?"
      answer: "I'll query rep performance data to rank all reps by average resolution time from fastest to slowest."
    - question: "Show me the trend of daily escalations"
      answer: "I'll query the daily support metrics for the full available date range and generate a line chart showing the escalation trend over time."""

tools = [search_support_cases, query_support_analytics, generate_chart]

graph = create_react_agent(
    llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print(f"LangGraph agent compiled. Nodes: {list(graph.get_graph().nodes.keys())}")

LangGraph agent compiled. Nodes: ['__start__', 'agent', 'tools', '__end__']


/var/folders/kt/jnpc5l790719m738576fttpm0000gn/T/ipykernel_6905/934488139.py:46: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = create_react_agent(


## 4. Test the Agent

In [12]:
response = graph.invoke(
    {"messages": [("user", "What is the average CSAT score by product?")]}
)
print(response["messages"][-1].content)

alt.Chart(...)

Here are the **average CSAT scores by product** (scale of 1–5), ranked highest to lowest:

| Rank | Product | Avg CSAT |
|------|---------------------|----------|
| 1 | AnalyticsHub | **3.89** |
| 2 | SecureVault | **3.84** |
| 3 | API Gateway | **3.82** |
| 4 | DataSync Pro | **3.79** |
| 5 | CloudStore Platform | **3.77** |

**Key Takeaways:**
- 📊 The scores are **tightly clustered** between 3.77 and 3.89 — a narrow range of just 0.12 points.
- 🏆 **AnalyticsHub** leads with the highest satisfaction at **3.89**.
- ⚠️ **CloudStore Platform** sits at the bottom at **3.77**, which may warrant attention.
- All products fall in the **3.7–3.9 range**, suggesting generally moderate satisfaction across the board with room for improvement.


In [ ]:
response = graph.invoke(
    {"messages": [("user", "plot a bar chart of the average CSAT score by product")]}
)
print(response["messages"][-1].content)

In [ ]:
response = graph.invoke(
    {"messages": [("user", "Find cases related to authentication failures and how they were resolved")]}
)
print(response["messages"][-1].content)

## 5. Instrument with TruLens + Snowflake AI Observability

We wrap the LangGraph agent with `TruGraph` which:
- Auto-instruments all graph nodes and tool calls
- Captures full execution traces (inputs, outputs, latency)
- Exports traces to Snowflake for the AI Observability UI

In [ ]:
APP_NAME = "LANGGRAPH_CUSTOMER_SUPPORT_AGENT"

try:
    session.sql(f'DROP EXTERNAL AGENT {APP_NAME}').collect()
except Exception:
    pass

[Row(status='LANGGRAPH_CUSTOMER_SUPPORT_AGENT successfully dropped.')]

In [ ]:
from trulens.apps.langgraph import TruGraph
from trulens.connectors.snowflake import SnowflakeConnector

tru_snowflake_connector = SnowflakeConnector(snowpark_session=session)

APP_NAME = "LANGGRAPH_CUSTOMER_SUPPORT_AGENT"
APP_VERSION = "ALIGNED_EVAL_V1"

tru_graph = TruGraph(
    graph,
    app_name=APP_NAME,
    app_version=APP_VERSION,
    connector=tru_snowflake_connector,
    main_method = graph.invoke
)

print(f"TruGraph registered: {APP_NAME} / {APP_VERSION}")

✅ experimental Feature.OTEL_TRACING enabled.
🔒 experimental Feature.OTEL_TRACING is enabled and cannot be changed.


/opt/anaconda3/envs/snow_tru_py314/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


instrumenting <class 'langgraph.graph.state.StateGraph'> for base <class 'langgraph.graph.state.StateGraph'>
instrumenting <class 'langgraph.graph.state.CompiledStateGraph'> for base <class 'langgraph.graph.state.CompiledStateGraph'>
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
	instrumenting astream_events
	instrumenting stream
	instrumenting astream
	instrumenting astream_events
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
	instrumenting stream_mode
instrumenting <class 'langgraph.graph.state.CompiledStateGraph'> for base <class 'langgraph.pregel.main.Pregel'>
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
	instrumenting astream_events
	instrumenting stream
	instrumenting astream
	instrumenting astream_events
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
	instrumenting stream_mode
TruGraph registered: LANGGRAP

## 6. Define Evaluation Dataset

We create a test dataset that exercises both tools across different query types.

In [15]:
import pandas as pd

queries_df = session.table("CUST_SUPPORT_DEMO.AGENTS.EVAL_DATA").to_pandas()

queries_df['query_json'] = queries_df['INPUT_QUERY'].apply(lambda x: {"messages": [("user", x)]})
queries_df['ground_truth_string'] = queries_df['GROUND_TRUTH_DATA'].apply(lambda x: json.loads(x).get('ground_truth_output'))
print(f"Evaluation dataset: {len(queries_df)} queries")
queries_df

Evaluation dataset: 25 queries


,INPUT_QUERY,GROUND_TRUTH_DATA,query_json,ground_truth_string
0,What is the average CSAT score by product?,"{\n ""ground_truth_output"": ""The average CSAT ...","{'messages': [('user', 'What is the average CS...",The average CSAT scores by product are: Analyt...
1,Which rep has the fastest average resolution t...,"{\n ""ground_truth_output"": ""Rachel Foster has...","{'messages': [('user', 'Which rep has the fast...",Rachel Foster has the fastest average resoluti...
2,What is the escalation rate for Critical prior...,"{\n ""ground_truth_output"": ""The escalation ra...","{'messages': [('user', 'What is the escalation...",The escalation rate for Critical priority case...
3,How many cases were there for each issue categ...,"{\n ""ground_truth_output"": ""Case counts by is...","{'messages': [('user', 'How many cases were th...",Case counts by issue category: Login/Authentic...
4,How many support cases were there for each pro...,"{\n ""ground_truth_output"": ""Cases by product:...","{'messages': [('user', 'How many support cases...","Cases by product: SecureVault: 47, CloudStore ..."
5,What is the average CSAT score by issue category?,"{\n ""ground_truth_output"": ""Average CSAT by i...","{'messages': [('user', 'What is the average CS...",Average CSAT by issue category: Data Loss: 3.9...
6,Which reps handled the most cases overall?,"{\n ""ground_truth_output"": ""Reps by total cas...","{'messages': [('user', 'Which reps handled the...","Reps by total cases handled: David Kim: 216, C..."
7,What is the escalation rate for each priority ...,"{\n ""ground_truth_output"": ""Escalation rates ...","{'messages': [('user', 'What is the escalation...","Escalation rates by priority: Critical: 20%, H..."
8,What is the overall average CSAT score across ...,"{\n ""ground_truth_output"": ""The overall avera...","{'messages': [('user', 'What is the overall av...",The overall average CSAT score across all 200 ...
9,What is the average first response time by pri...,"{\n ""ground_truth_output"": ""Average first res...","{'messages': [('user', 'What is the average fi...",Average first response time by priority: Low: ...


## 7. Execute Evaluation Run

In [ ]:
from trulens.core.run import Run
from trulens.core.run import RunConfig
import datetime

run_config = RunConfig(
    run_name="LANGGRAPH_ALIGNED_EVAL_V1",
    description="Aligned evaluation - correctness, groundedness, coherence, logical_consistency",
    dataset_name="TEST_QUERIES_DF",
    source_type="DATAFRAME",
    label="LANGGRAPH_ALIGNED",
    dataset_spec={
        "RECORD_ROOT.INPUT": "query_json",
        "RECORD_ROOT.GROUND_TRUTH_OUTPUT": "ground_truth_string",
    },
)

In [17]:
run = tru_graph.add_run(run_config=run_config)

run.start(input_df=queries_df)

alt.Chart(...)

Evaluator thread encountered an error: 'str' object has no attribute 'get'


alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

...

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

## 8. Compute Evaluation Metrics

We compute three key RAG evaluation metrics:
- **Answer Relevance**: Is the response relevant to the user's query?
- **Context Relevance**: Is the retrieved context relevant to the query?
- **Groundedness**: Is the response grounded in the retrieved context?

In [ ]:
import time
from trulens.core import Metric, Selector
from trulens.providers.cortex import Cortex

provider = Cortex(snowpark_session=session, model_engine="claude-sonnet-4-6")

logical_consistency_metric = Metric(
    name="logical_consistency",
    implementation=provider.logical_consistency_with_cot_reasons,
    metric_type="logical_consistency",
    description="Evaluates logical consistency of agent reasoning across the full execution trace",
    selectors={
        "trace": Selector(trace_level=True),
    },
)

metric_list = [
    "correctness",
    "groundedness",
    "coherence",
    logical_consistency_metric,
]

run.compute_metrics(metric_list)
print("Metrics computation started...")

while run.get_status() not in ("COMPLETED", "PARTIALLY_COMPLETED", "CANCELLED"):
    print(f"  Status: {run.get_status()}")
    time.sleep(5)

print(f"Evaluation complete. Final status: {run.get_status()}")

Metrics computation started...
  Status: RunStatus.INVOCATION_COMPLETED
  Status: RunStatus.INVOCATION_COMPLETED
  Status: RunStatus.COMPUTATION_IN_PROGRESS
  Status: RunStatus.COMPUTATION_IN_PROGRESS
  Status: RunStatus.COMPUTATION_IN_PROGRESS
  Status: RunStatus.COMPUTATION_IN_PROGRESS
Evaluation complete. Final status: RunStatus.COMPLETED


## 9. View Results

Results are available in the **Snowsight AI Observability UI**:

1. Navigate to **AI & ML > Evaluations** in Snowsight
2. Filter by app name: `customer_support_langgraph_agent`
3. Select the run to inspect individual traces and metrics

You can also query the results programmatically:

In [19]:
print(f"Run Name:   {run_config.run_name}")
print(f"App Name:   {APP_NAME}")
print(f"App Version: {APP_VERSION}")
print(f"Final Status: {run.get_status()}")
print()
print("View full results in Snowsight:")
print("  → AI & ML > Evaluations")
print(f"  → Look for app '{APP_NAME}' version '{APP_VERSION}'")

Run Name:   LANGGRAPH_SUPPORT_AGENT_EVAL_RUN
App Name:   LANGGRAPH_CUSTOMER_SUPPORT_AGENT
App Version: CHARTING_ENABLED
Final Status: RunStatus.COMPLETED

View full results in Snowsight:
  → AI & ML > Evaluations
  → Look for app 'LANGGRAPH_CUSTOMER_SUPPORT_AGENT' version 'CHARTING_ENABLED'


In [21]:
eval_df = session.sql(f"""
    SELECT *
    FROM TABLE(
        SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
            'CUST_SUPPORT_DEMO', 'AGENTS', '{APP_NAME}', 'EXTERNAL AGENT', '{run_config.run_name}'
        )
    )
    ORDER BY TIMESTAMP DESC
    LIMIT 50
""").to_pandas()

print(f"Evaluation records: {len(eval_df)}")
if not eval_df.empty:
    display(eval_df)
else:
    print("No evaluation data yet. Results may take a moment to appear.")
    print("Check Snowsight → AI & ML → Evaluations for the latest results.")

Evaluation records: 50


,RECORD_ID,INPUT_ID,REQUEST_ID,TIMESTAMP,DURATION_MS,INPUT,OUTPUT,ERROR,GROUND_TRUTH,METRIC_NAME,EVAL_AGG_SCORE,METRIC_TYPE,METRIC_STATUS,METRIC_CALLS,TOTAL_INPUT_TOKENS,TOTAL_OUTPUT_TOKENS,LLM_CALL_COUNT
0,78e29dd3-cc7d-451e-ab14-186af0d76465,obj_hash_0becefd44dbbbd2f46beb29edf946077,None,2026-04-13 22:22:44.113170-07:00,25190,What is the resolution rate for each rep?,"Here's the resolution rate for all 10 reps, co...",None,"Resolution rates by rep: James Wilson: 83.9%, ...",groundedness,0.369048,groundedness,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""The statement should ...",0,0,0
1,78e29dd3-cc7d-451e-ab14-186af0d76465,obj_hash_0becefd44dbbbd2f46beb29edf946077,None,2026-04-13 22:22:44.113170-07:00,25190,What is the resolution rate for each rep?,"Here's the resolution rate for all 10 reps, co...",None,"Resolution rates by rep: James Wilson: 83.9%, ...",coherence,1.000000,coherence,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""1. Clarity of data pr...",0,0,0
2,78e29dd3-cc7d-451e-ab14-186af0d76465,obj_hash_0becefd44dbbbd2f46beb29edf946077,None,2026-04-13 22:22:44.113170-07:00,25190,What is the resolution rate for each rep?,"Here's the resolution rate for all 10 reps, co...",None,"Resolution rates by rep: James Wilson: 83.9%, ...",correctness,1.000000,correctness,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""- The actual response...",0,0,0
3,fd9b2c16-bce6-447a-94c3-f071d51ee831,obj_hash_6e171ec90e35a0454ff7191e805cd20d,None,2026-04-13 22:22:18.920056-07:00,10269,Search for cases related to TLS or security vu...,"Here's a summary of the cases found, grouped b...",None,Cases include vulnerability scans showing outd...,coherence,1.000000,coherence,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""1. Clarity of issue d...",0,0,0
4,fd9b2c16-bce6-447a-94c3-f071d51ee831,obj_hash_6e171ec90e35a0454ff7191e805cd20d,None,2026-04-13 22:22:18.920056-07:00,10269,Search for cases related to TLS or security vu...,"Here's a summary of the cases found, grouped b...",None,Cases include vulnerability scans showing outd...,groundedness,0.684211,groundedness,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""The statement should ...",0,0,0
5,fd9b2c16-bce6-447a-94c3-f071d51ee831,obj_hash_6e171ec90e35a0454ff7191e805cd20d,None,2026-04-13 22:22:18.920056-07:00,10269,Search for cases related to TLS or security vu...,"Here's a summary of the cases found, grouped b...",None,Cases include vulnerability scans showing outd...,correctness,0.666667,correctness,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""- Relevance to TLS or...",0,0,0
6,36a88bef-d64c-4f0c-bcd2-16c769d44a60,obj_hash_127427b3b8b79ee358ea21d71f2a0609,None,2026-04-13 22:22:08.650603-07:00,9814,Find cases about API response time or timeout ...,"Here are the API-related cases found, grouped ...",None,Cases include API response times spiking durin...,groundedness,0.916667,groundedness,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""The statement should ...",0,0,0
7,36a88bef-d64c-4f0c-bcd2-16c769d44a60,obj_hash_127427b3b8b79ee358ea21d71f2a0609,None,2026-04-13 22:22:08.650603-07:00,9814,Find cases about API response time or timeout ...,"Here are the API-related cases found, grouped ...",None,Cases include API response times spiking durin...,coherence,1.000000,coherence,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""1. Clarity of issue c...",0,0,0
8,36a88bef-d64c-4f0c-bcd2-16c769d44a60,obj_hash_127427b3b8b79ee358ea21d71f2a0609,None,2026-04-13 22:22:08.650603-07:00,9814,Find cases about API response time or timeout ...,"Here are the API-related cases found, grouped ...",None,Cases include API response times spiking durin...,correctness,0.666667,correctness,"{\n ""code"": null,\n ""message"": null\n}","[\n {\n ""criteria"": ""- Relevance to API re...",0,0,0
9,ca5ea888-8ced-481a-8738-079ff3c655d5,obj_hash_5021514842c40f4a8f10f

In [23]:
obs_event_sql = f"""SELECT * FROM TABLE(
    SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CUST_SUPPORT_DEMO', 
        'AGENTS', 
        '{APP_NAME}', 
        'EXTERNAL AGENT')
) 
WHERE RECORD_ATTRIBUTES:"snow.ai.observability.run.name" = '{run_config.run_name}';"""

df = session.sql(obs_event_sql).to_pandas()

df

,TIMESTAMP,START_TIMESTAMP,TRACE,RESOURCE_ATTRIBUTES,RECORD_TYPE,RECORD,RECORD_ATTRIBUTES,VALUE
0,2026-04-14 05:24:09.293958741,2026-04-14 05:24:09.291525695,"{\n ""span_id"": ""336675465033796e""\n}","{\n ""db.user"": ""SYSTEM"",\n ""snow.database.id...",SPAN,"{\n ""kind"": ""SPAN_KIND_UNSPECIFIED"",\n ""name...","{\n ""ai.observability.eval.args"": {\n ""ai....",None
1,2026-04-14 05:24:09.285668278,2026-04-14 05:24:09.274161297,"{\n ""span_id"": ""7775517571373265""\n}","{\n ""db.user"": ""SYSTEM"",\n ""snow.database.id...",SPAN,"{\n ""kind"": ""SPAN_KIND_UNSPECIFIED"",\n ""name...","{\n ""ai.observability.eval.eval_root_id"": ""wu...",None
2,2026-04-14 05:24:09.315318336,2026-04-14 05:24:09.314993610,"{\n ""span_id"": ""394b463279334365""\n}","{\n ""db.user"": ""SYSTEM"",\n ""snow.database.id...",SPAN,"{\n ""kind"": ""SPAN_KIND_UNSPECIFIED"",\n ""name...","{\n ""ai.observability.eval.args"": {\n ""ai....",None
3,2026-04-14 05:24:09.314486209,2026-04-14 05:24:09.314162698,"{\n ""span_id"": ""31444d676850586d""\n}","{\n ""db.user"": ""SYSTEM"",\n ""snow.database.id...",SPAN,"{\n ""kind"": ""SPAN_KIND_UNSPECIFIED"",\n ""name...","{\n ""ai.observability.eval.eval_root_id"": ""1D...",None
4,2026-04-14 05:24:09.316960977,2026-04-14 05:24:09.316876007,"{\n ""span_id"": ""414536506759375a""\n}","{\n ""db.user"": ""SYSTEM"",\n ""snow.database.id...",SPAN,"{\n ""kind"": ""SPAN_KIND_UNSPECIFIED"",\n ""name...","{\n ""ai.observability.eval.args"": {\n ""ai....",None
...,...,...,...,...,...,...,...,...
867,2026-04-14 05:21:58.833726000,2026-04-14 05:21:58.833068000,"{\n ""span_id"": ""f2fd541daf21aae1"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_4979...",None
868,2026-04-14 05:21:11.302706000,2026-04-14 05:21:02.990499000,"{\n ""span_id"": ""255c56db182de30e"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_4979...",None
869,2026-04-14 05:21:11.305007000,2026-04-14 05:21:11.304086000,"{\n ""span_id"": ""5b009b9330ca5525"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_4979...",None
870,2026-04-14 05:21:11.306372000,2026-04-14 05:20:58.943604000,"{\n ""span_id"": ""296be831dcfaa502"",\n ""trace_...","{\n ""db.user"": ""EBOTWICK"",\n ""snow.database....",SPAN,"{\n ""kind"": ""SPAN_KIND_INTERNAL"",\n ""name"": ...","{\n ""ai.observability.app_id"": ""app_hash_4979...",None


## Next Steps

- **Iterate on the agent**: Try different LLMs (`llama3.1-70b`, `mistral-large`), adjust the system prompt, or add more tools
- **Compare versions**: Create new app versions and compare evaluation runs side-by-side in Snowsight
- **Add ground truth**: Include expected answers in the dataset to compute the `correctness` metric
- **Add more metrics**: Compute `coherence` or custom metrics for domain-specific evaluation
- **Production deployment**: Use the best-performing configuration to deploy as a Cortex Agent or Streamlit app